In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import random

# Paths
TRAIN_IMAGES = "/teamspace/studios/this_studio/cell_dataset/train/train_images"
TRAIN_MASKS  = "/teamspace/studios/this_studio/cell_dataset/train/train_masks"
TEST_IMAGES  = "/teamspace/studios/this_studio/cell_dataset/test/test_images"

NUM_CLASSES = 5

# Confirm GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
COLOR_TO_CLASS = {
    (0,   0,   0):   0, 
    (255, 255, 0):   1, 
    (255, 0,   0):   2, 
    (0,   255, 0):   3, 
    (0,   0,   255): 4, 
}
CLASS_TO_COLOR = {v: k for k, v in COLOR_TO_CLASS.items()}

def rgb_to_mask(mask_rgb):
    H, W, _ = mask_rgb.shape
    label_map = np.zeros((H, W), dtype=np.int64)
    for color, class_id in COLOR_TO_CLASS.items():
        match = np.all(mask_rgb == np.array(color), axis=-1)
        label_map[match] = class_id
    return label_map

def torchvision_random_crop_params(img, scale=(0.7, 1.0)):
    width, height = img.size
    area = height * width
    target_area = random.uniform(scale[0], scale[1]) * area
    aspect_ratio = width / height
    w = int(round((target_area * aspect_ratio) ** 0.5))
    h = int(round((target_area / aspect_ratio) ** 0.5))
    w = min(w, width)
    h = min(h, height)
    i = random.randint(0, height - h)
    j = random.randint(0, width  - w)
    return i, j, h, w

def augment(image, mask):
    if random.random() > 0.5:
        image = TF.hflip(image)
        mask  = TF.hflip(mask)
    if random.random() > 0.5:
        image = TF.vflip(image)
        mask  = TF.vflip(mask)
    angle = random.choice([0, 90, 180, 270])
    image = TF.rotate(image, angle)
    mask  = TF.rotate(mask,  angle)
    i, j, h, w = torchvision_random_crop_params(image, scale=(0.7, 1.0))
    image = TF.resized_crop(image, i, j, h, w, size=(512, 512),
                             interpolation=TF.InterpolationMode.BILINEAR)
    mask  = TF.resized_crop(mask,  i, j, h, w, size=(512, 512),
                             interpolation=TF.InterpolationMode.NEAREST)
    image = TF.adjust_brightness(image, brightness_factor=random.uniform(0.8, 1.2))
    image = TF.adjust_contrast(image,   contrast_factor=random.uniform(0.8, 1.2))
    return image, mask

class CellSegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, filenames, is_train=True):
        self.image_dir = image_dir
        self.mask_dir  = mask_dir
        self.filenames = filenames
        self.is_train  = is_train
    def __len__(self):
        return len(self.filenames)
    def __getitem__(self, idx):
        fname = self.filenames[idx]
        image = Image.open(os.path.join(self.image_dir, fname)).convert("RGB")
        mask  = Image.open(os.path.join(self.mask_dir,  fname)).convert("RGB")
        if self.is_train:
            image, mask = augment(image, mask)
        image = TF.to_tensor(image)
        image = TF.normalize(image,
                             mean=[0.485, 0.456, 0.406],
                             std= [0.229, 0.224, 0.225])
        mask = np.array(mask)
        mask = rgb_to_mask(mask)
        mask = torch.from_numpy(mask).long()
        return image, mask

In [3]:
all_images = sorted(os.listdir(TRAIN_IMAGES))
all_masks  = sorted(os.listdir(TRAIN_MASKS))

train_imgs, val_imgs, train_msks, val_msks = train_test_split(
    all_images, all_masks,
    test_size=0.1,
    random_state=42
)

train_dataset = CellSegmentationDataset(TRAIN_IMAGES, TRAIN_MASKS, train_imgs, is_train=True)
val_dataset   = CellSegmentationDataset(TRAIN_IMAGES, TRAIN_MASKS, val_imgs,   is_train=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders ready!")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

DataLoaders ready!
Train batches: 25
Val batches:   3


In [4]:
class ConvBlock(nn.Module):
    """
    The basic building block of UNet.
    Two consecutive Conv→BatchNorm→ReLU operations.

    Why two convolutions?
    - First conv learns low-level features (edges, textures)
    - Second conv combines them into higher-level patterns
    - BatchNorm stabilizes training and acts as regularization
    - ReLU introduces non-linearity — without it the network
      is just a linear transformation regardless of depth
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            # Conv 1
            nn.Conv2d(in_channels, out_channels,
                      kernel_size=3, padding=1, bias=False),
            # padding=1 keeps spatial dimensions same (512→512)
            # bias=False because BatchNorm already has a learnable bias
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            # Conv 2
            nn.Conv2d(out_channels, out_channels,
                      kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = ConvBlock(3,   64)   
        self.enc2 = ConvBlock(64,  128)  
        self.enc3 = ConvBlock(128, 256)  
        self.enc4 = ConvBlock(256, 512)  

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        s1 = self.enc1(x)     
        s2 = self.enc2(self.pool(s1)) 
        s3 = self.enc3(self.pool(s2)) 
        s4 = self.enc4(self.pool(s3)) 

        return self.pool(s4), [s1, s2, s3, s4]


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.up3 = nn.ConvTranspose2d(512,  256, kernel_size=2, stride=2)
        self.up2 = nn.ConvTranspose2d(256,  128, kernel_size=2, stride=2)
        self.up1 = nn.ConvTranspose2d(128,  64,  kernel_size=2, stride=2)

        self.dec4 = ConvBlock(1024, 512)
        self.dec3 = ConvBlock(512,  256)
        self.dec2 = ConvBlock(256,  128)
        self.dec1 = ConvBlock(128,  64)

    def forward(self, x, skips):
        s1, s2, s3, s4 = skips

        x = self.up4(x)              
        x = torch.cat([x, s4], dim=1)
        x = self.dec4(x)             

        x = self.up3(x)              
        x = torch.cat([x, s3], dim=1)
        x = self.dec3(x)             

        x = self.up2(x)              
        x = torch.cat([x, s2], dim=1)
        x = self.dec2(x)             

        x = self.up1(x)              
        x = torch.cat([x, s1], dim=1)
        x = self.dec1(x)             

        return x


class UNet(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()

        self.encoder    = Encoder()
        self.bottleneck = ConvBlock(512, 1024)

        self.decoder    = Decoder()
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        x, skips = self.encoder(x)
        x = self.bottleneck(x)      
        x = self.decoder(x, skips)  
        return self.final_conv(x)   

model = UNet(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"UNet initialized!")
print(f"Total parameters: {total_params:,}")

# Test with a dummy input to verify shapes
dummy = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")


UNet initialized!
Total parameters: 31,037,893
Input shape:  torch.Size([1, 3, 512, 512])
Output shape: torch.Size([1, 5, 512, 512])


In [5]:
class CombinedLoss(nn.Module):
    """
    Combined Dice Loss + Weighted Cross Entropy Loss.

    Why combine two losses?
    - Cross Entropy: penalizes wrong predictions pixel by pixel
    - Dice Loss: measures overlap between prediction and ground truth
                 naturally handles class imbalance by treating each
                 class equally regardless of pixel count

    Using both together gives better results than either alone —
    Cross Entropy provides stable gradients early in training,
    Dice Loss ensures minority classes (Yellow = 0.2%) are not ignored.

    Final loss = CrossEntropy + Dice Loss
    """
    def __init__(self, weights=None):
        super().__init__()

        # Class weights for Cross Entropy
        # Higher weight = model penalized more for mistakes on that class
        # We set very high weight for Yellow Cell since it's only 0.2%
        if weights is None:
            weights = torch.tensor(
                [0.1,   # Background  — very common, low weight
                 10.0,  # Yellow Cell — extremely rare, very high weight
                 3.0,   # Red Cell
                 1.0,   # Green Cell  — most common cell, lower weight
                 2.0],  # Blue Cell
                dtype=torch.float32
            ).to(device)

        # CrossEntropyLoss expects raw logits (no softmax needed)
        # weight parameter assigns different importance to each class
        self.ce_loss = nn.CrossEntropyLoss(weight=weights)

    def dice_loss(self, preds, targets, smooth=1e-6):
        """
        Dice Loss for multi-class segmentation.

        Dice coefficient measures overlap between prediction and ground truth:
        Dice = 2 * |Pred ∩ Target| / (|Pred| + |Target|)

        Dice = 1.0 → perfect overlap
        Dice = 0.0 → no overlap

        Dice Loss = 1 - Dice (so minimizing loss = maximizing overlap)

        smooth: small value to avoid division by zero when a class
                is completely absent from a batch
        """
        # Convert logits to probabilities using softmax
        # dim=1 because channels (classes) are dimension 1
        preds_soft = torch.softmax(preds, dim=1)

        # Convert class IDs to one-hot encoding
        # e.g. class 3 → [0, 0, 0, 1, 0]
        # one_hot shape: (B, H, W, C) → permute to (B, C, H, W)
        targets_one_hot = torch.nn.functional.one_hot(
            targets, num_classes=NUM_CLASSES
        ).permute(0, 3, 1, 2).float()

        # Compute Dice per class then average
        # sum over H and W dimensions (2 and 3)
        intersection = (preds_soft * targets_one_hot).sum(dim=(2, 3))
        union        = preds_soft.sum(dim=(2, 3)) + targets_one_hot.sum(dim=(2, 3))

        dice_per_class = (2.0 * intersection + smooth) / (union + smooth)

        # Average across classes and batch
        return 1.0 - dice_per_class.mean()

    def forward(self, preds, targets):
        ce   = self.ce_loss(preds, targets)
        dice = self.dice_loss(preds, targets)

        # Equal weighting of both losses
        return ce + dice

criterion = CombinedLoss()
print("✅ Loss function defined!")

# Test loss with dummy data
dummy_preds   = torch.randn(4, 5, 512, 512).to(device)
dummy_targets = torch.randint(0, 5, (4, 512, 512)).to(device)
loss = criterion(dummy_preds, dummy_targets)
print(f"Test loss value: {loss.item():.4f}")

✅ Loss function defined!
Test loss value: 2.7747


In [6]:
# ── Optimizer ─────────────────────────────────────────────────────────
# Adam is the standard choice for segmentation tasks
# It adapts the learning rate for each parameter individually
# lr=1e-4 is a good starting point for medical image segmentation
# weight_decay adds L2 regularization — penalizes large weights
# to prevent overfitting on our small 97-image dataset
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# ── Learning Rate Scheduler ───────────────────────────────────────────
# ReduceLROnPlateau monitors validation loss
# If validation loss doesn't improve for 'patience' epochs,
# it multiplies the learning rate by 'factor'
# This helps the model escape plateaus and fine-tune more carefully
#
# Example:
# Epoch 1-5:  lr = 0.0001  (no improvement detected)
# Epoch 6:    lr = 0.00005 (reduced by factor 0.5)
# Epoch 7-11: lr = 0.00005 (no improvement)
# Epoch 12:   lr = 0.000025 (reduced again)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",       # monitor validation loss (minimize)
    factor=0.5,       # multiply lr by 0.5 when triggered
    patience=5     # print when lr is reduced
)

print("Optimizer and scheduler defined!")
print(f"Initial learning rate: {optimizer.param_groups[0]['lr']}")

Optimizer and scheduler defined!
Initial learning rate: 0.0001


In [7]:
def compute_metrics(preds, targets, num_classes=5, smooth=1e-6):
    """
    Compute mIoU and Dice Score for a batch of predictions.

    mIoU (Mean Intersection over Union):
    - IoU per class = overlap / union of predicted and true pixels
    - mIoU = average IoU across all non-background classes
    - IoU = 1.0 → perfect prediction
    - IoU = 0.0 → no overlap at all

    Dice Score:
    - Similar to IoU but uses a different formula
    - Dice = 2 * overlap / (pred_size + true_size)
    - Always slightly higher than IoU for same prediction

    We exclude background (class 0) from both metrics
    because it dominates at 83.6% and would inflate scores
    """
    # Convert logits → predicted class per pixel
    # argmax picks the class with highest score for each pixel
    preds = torch.argmax(preds, dim=1)  # (B, H, W)

    iou_per_class  = []
    dice_per_class = []

    # Compute metrics for each non-background class
    for cls in range(1, num_classes):  # skip class 0 (background)
        pred_cls   = (preds   == cls)  # boolean mask: where did we predict cls?
        target_cls = (targets == cls)  # boolean mask: where is cls in ground truth?

        # Intersection: pixels correctly predicted as cls
        intersection = (pred_cls & target_cls).sum().float()

        # Union: all pixels predicted OR true for cls
        union = (pred_cls | target_cls).sum().float()

        # Dice denominator
        pred_sum   = pred_cls.sum().float()
        target_sum = target_cls.sum().float()

        # IoU for this class
        iou  = (intersection + smooth) / (union + smooth)

        # Dice for this class
        dice = (2.0 * intersection + smooth) / (pred_sum + target_sum + smooth)

        iou_per_class.append(iou.item())
        dice_per_class.append(dice.item())

    miou       = np.mean(iou_per_class)
    mean_dice  = np.mean(dice_per_class)

    return miou, mean_dice

print("Metrics function defined!")

Metrics function defined!


In [8]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Run one full pass through the training data.
    Returns average loss for the epoch.
    """
    model.train()  # Enable dropout and batch norm training mode
    total_loss = 0.0

    for batch_idx, (images, masks) in enumerate(loader):
        # Move data to GPU
        images = images.to(device)
        masks  = masks.to(device)

        # ── Forward pass ──────────────────────────────────────────────
        # Pass images through UNet to get predictions
        # preds shape: (B, 5, 512, 512) — raw logits
        preds = model(images)

        # ── Compute loss ──────────────────────────────────────────────
        loss = criterion(preds, masks)

        # ── Backward pass ─────────────────────────────────────────────
        optimizer.zero_grad()  # Clear gradients from previous batch
        loss.backward()        # Compute gradients via backpropagation
        optimizer.step()       # Update model weights

        total_loss += loss.item()

    return total_loss / len(loader)  # Average loss per batch


def validate(model, loader, criterion, device):
    """
    Evaluate model on validation set.
    No gradient computation needed — saves memory and speeds up evaluation.
    Returns average loss, mIoU and Dice Score.
    """
    model.eval()  # Disable dropout, use running stats for batch norm
    total_loss = 0.0
    all_miou   = []
    all_dice   = []

    with torch.no_grad():  # No gradients needed for evaluation
        for images, masks in loader:
            images = images.to(device)
            masks  = masks.to(device)

            preds = model(images)
            loss  = criterion(preds, masks)

            total_loss += loss.item()

            # Compute metrics for this batch
            miou, dice = compute_metrics(preds, masks)
            all_miou.append(miou)
            all_dice.append(dice)

    avg_loss = total_loss / len(loader)
    avg_miou = np.mean(all_miou)
    avg_dice = np.mean(all_dice)

    return avg_loss, avg_miou, avg_dice

print("Training functions defined!")

Training functions defined!


In [9]:
# ── Training Configuration ────────────────────────────────────────────
NUM_EPOCHS  = 200
BATCH_SIZE  = 8
SAVE_PATH   = "../models/unet_best.pth"
PATIENCE    = 20  # stop if no improvement for 20 epochs
os.makedirs("../models", exist_ok=True)

# ── Recreate DataLoaders with new batch size ──────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Updated train batches: {len(train_loader)}")
print(f"Updated val batches:   {len(val_loader)}")

# ── Training History ──────────────────────────────────────────────────
history = {
    "train_loss": [],
    "val_loss":   [],
    "val_miou":   [],
    "val_dice":   []
}

best_val_loss = float("inf")
best_epoch    = 0
no_improve    = 0

print("🚀 Starting training...")
print(f"   Epochs:     {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Device:     {device}")
print(f"   Early stop: {PATIENCE} epochs patience")
print("=" * 65)

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Train ─────────────────────────────────────────────────────────
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # ── Validate ──────────────────────────────────────────────────────
    val_loss, val_miou, val_dice = validate(model, val_loader, criterion, device)

    # ── Scheduler step ────────────────────────────────────────────────
    scheduler.step(val_loss)

    # ── Save best model + Early Stopping ──────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch
        no_improve    = 0
        torch.save(model.state_dict(), SAVE_PATH)
        saved_str = "✅ saved"
    else:
        no_improve += 1
        saved_str  = f"no improve {no_improve}/{PATIENCE}"

    # ── Log progress ──────────────────────────────────────────────────
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_miou"].append(val_miou)
    history["val_dice"].append(val_dice)

    print(f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"mIoU: {val_miou:.4f} | "
          f"Dice: {val_dice:.4f} | "
          f"{saved_str}")

    # ── Early Stopping Check ──────────────────────────────────────────
    if no_improve >= PATIENCE:
        print(f"\n⚠️  Early stopping triggered at epoch {epoch}")
        print(f"   No improvement for {PATIENCE} consecutive epochs")
        break

print("=" * 65)
print(f" Training complete!")
print(f"   Best model saved at Epoch {best_epoch}")
print(f"   Best val loss:  {best_val_loss:.4f}")
print(f"   Final mIoU:     {history['val_miou'][best_epoch-1]:.4f}")
print(f"   Final Dice:     {history['val_dice'][best_epoch-1]:.4f}")

Updated train batches: 13
Updated val batches:   2
🚀 Starting training...
   Epochs:     200
   Batch size: 8
   Device:     cuda
   Early stop: 20 epochs patience
Epoch 001/200 | Train Loss: 2.3761 | Val Loss: 2.4000 | mIoU: 0.0272 | Dice: 0.0491 | ✅ saved
Epoch 002/200 | Train Loss: 2.1347 | Val Loss: 2.1938 | mIoU: 0.0681 | Dice: 0.1194 | ✅ saved
Epoch 003/200 | Train Loss: 2.0850 | Val Loss: 1.9802 | mIoU: 0.1285 | Dice: 0.2081 | ✅ saved
Epoch 004/200 | Train Loss: 2.0070 | Val Loss: 2.0007 | mIoU: 0.1266 | Dice: 0.2092 | no improve 1/20
Epoch 005/200 | Train Loss: 1.9030 | Val Loss: 1.8930 | mIoU: 0.1544 | Dice: 0.2527 | ✅ saved
Epoch 006/200 | Train Loss: 1.8524 | Val Loss: 1.8925 | mIoU: 0.1509 | Dice: 0.2464 | ✅ saved
Epoch 007/200 | Train Loss: 1.8878 | Val Loss: 1.9153 | mIoU: 0.1133 | Dice: 0.1876 | no improve 1/20
Epoch 008/200 | Train Loss: 1.7842 | Val Loss: 1.8490 | mIoU: 0.1543 | Dice: 0.2498 | ✅ saved
Epoch 009/200 | Train Loss: 1.8115 | Val Loss: 1.6966 | mIoU: 0.1743

In [10]:
torch.cuda.empty_cache()
import gc
gc.collect()
print(f"GPU memory freed!")

GPU memory freed!


In [11]:
import os
print(os.path.exists("../models/unet_best.pth"))
print(f"Size: {os.path.getsize('../models/unet_best.pth') / 1024**2:.1f} MB")

True
Size: 118.5 MB


In [12]:
# Load checkpoint and verify which run it belongs to
model_check = UNet(num_classes=NUM_CLASSES).to(device)
model_check.load_state_dict(torch.load("../models/unet_best.pth", map_location=device))

val_loss, val_miou, val_dice = validate(model_check, val_loader, criterion, device)
print(f"Val Loss: {val_loss:.4f} | mIoU: {val_miou:.4f} | Dice: {val_dice:.4f}")
print()
print("Run 1 (best): Val Loss ~1.2165 | mIoU ~0.3023 | Dice ~0.4411")
print("Run 3:        Val Loss ~1.3175 | mIoU ~0.2799 | Dice ~0.4158")

Val Loss: 1.3175 | mIoU: 0.2799 | Dice: 0.4158

Run 1 (best): Val Loss ~1.2165 | mIoU ~0.3023 | Dice ~0.4411
Run 3:        Val Loss ~1.3175 | mIoU ~0.2799 | Dice ~0.4158


Retrain with all improvements. We'll modify 3 things — loss function, learning rate, and add dropout to UNet.

In [13]:
model.load_state_dict(torch.load("../models/unet_best.pth", map_location=device))
print("Run 3 checkpoint loaded!")

Run 3 checkpoint loaded!


In [14]:
# Switch to Focal Loss
criterion = CombinedLoss()  # make sure this uses FocalLoss not CE

# Lower LR since model is partially trained
optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=7
)
print(f"✅ Focal Loss + LR {optimizer.param_groups[0]['lr']}")

✅ Focal Loss + LR 1e-05


In [15]:
NUM_EPOCHS    = 200
BATCH_SIZE    = 8
SAVE_PATH     = "../models/unet_best.pth"
PATIENCE      = 25  # more patience this time
best_val_loss = 1.3175  # Run 3's best
best_epoch    = 58
no_improve    = 0
history = {"train_loss": [], "val_loss": [], "val_miou": [], "val_dice": []}

print("🚀 Resuming from Run 3 checkpoint with Focal Loss...")
print(f"   Target to beat: Val Loss 1.3175 | mIoU 0.2799")
print(f"   LR: {optimizer.param_groups[0]['lr']}")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_miou, val_dice = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch + 58
        no_improve    = 0
        torch.save(model.state_dict(), SAVE_PATH)
        status = "✅ saved"
    else:
        no_improve += 1
        status = f"no improve {no_improve}/{PATIENCE}"

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_miou"].append(val_miou)
    history["val_dice"].append(val_dice)

    print(f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
          f"Train: {train_loss:.4f} | "
          f"Val: {val_loss:.4f} | "
          f"mIoU: {val_miou:.4f} | "
          f"Dice: {val_dice:.4f} | "
          f"{status} | LR: {current_lr:.2e}")

    if no_improve >= PATIENCE:
        print(f"\n⚠️  Early stopping at epoch {epoch}")
        break

print("=" * 70)
print(f"Done! Best epoch: {best_epoch} | Val Loss: {best_val_loss:.4f}")

🚀 Resuming from Run 3 checkpoint with Focal Loss...
   Target to beat: Val Loss 1.3175 | mIoU 0.2799
   LR: 1e-05
Epoch 001/200 | Train: 1.3154 | Val: 1.3453 | mIoU: 0.2829 | Dice: 0.4291 | no improve 1/25 | LR: 1.00e-05
Epoch 002/200 | Train: 1.3287 | Val: 1.3754 | mIoU: 0.2751 | Dice: 0.4233 | no improve 2/25 | LR: 1.00e-05
Epoch 003/200 | Train: 1.3096 | Val: 1.3158 | mIoU: 0.2798 | Dice: 0.4160 | ✅ saved | LR: 1.00e-05
Epoch 004/200 | Train: 1.2900 | Val: 1.3297 | mIoU: 0.2762 | Dice: 0.4156 | no improve 1/25 | LR: 1.00e-05
Epoch 005/200 | Train: 1.2983 | Val: 1.3684 | mIoU: 0.2840 | Dice: 0.4320 | no improve 2/25 | LR: 1.00e-05
Epoch 006/200 | Train: 1.3078 | Val: 1.3197 | mIoU: 0.2959 | Dice: 0.4474 | no improve 3/25 | LR: 1.00e-05
Epoch 007/200 | Train: 1.3023 | Val: 1.3023 | mIoU: 0.3082 | Dice: 0.4595 | ✅ saved | LR: 1.00e-05
Epoch 008/200 | Train: 1.4050 | Val: 1.3445 | mIoU: 0.2829 | Dice: 0.4322 | no improve 1/25 | LR: 1.00e-05
Epoch 009/200 | Train: 1.3202 | Val: 1.3452 | 